In [1]:
import argparse
from pathlib import Path
import sys

import torch 

import numpy as np
import pandas as pd
import yaml
from sklearn.datasets import fetch_openml
from src.io.read_file import merged_specs
from src.data.tabular import (
    create_dataloader,
    get_num_cat_columns,
    preprocess_features,
    remove_constant_columns,
    remove_duplicate_rows,
    remove_identifier_columns,
    remove_missing_label,
    split_data,
)

In [6]:
specs = merged_specs(Path("../databank/tabular_datasets.yaml"))
names = list(specs)

In [7]:
"wine" in names
spec = specs["wine"]

In [9]:
dataset = fetch_openml(data_id=spec["data_id"], as_frame=True, parser="auto")

X, y = dataset.data.copy(), dataset.target.copy()
y = pd.Series(y, index=X.index)
print("original Feature shape", X.shape, y.shape, type(X), type(y))


original Feature shape (178, 13) (178,) <class 'pandas.DataFrame'> <class 'pandas.Series'>


In [10]:
X, y, duplicate_count = remove_duplicate_rows(X, y)
X, y, missing_label_count = remove_missing_label(X, y)
X = remove_identifier_columns(X, spec.get("identifier_columns", []))

print("Shape after removing identifiers", X.shape, y.shape)

Shape after removing identifiers (178, 13) (178,)


In [11]:
expected_features = spec.get("expected_features")
if expected_features is not None and X.shape[1] != expected_features:
    raise AssertionError(
        f"expected {expected_features} predictors, received {X.shape[1]}"
    )

X, constant_columns = remove_constant_columns(X)

print("shape after removing constant columns", X.shape, y.shape)

shape after removing constant columns (178, 13) (178,)


In [12]:
X, numeric, categorical = get_num_cat_columns(
    X,
    categorical_columns=spec.get("manual_categorical_columns", []),
    numeric_columns=spec.get("manual_numeric_columns", []),
)

print("cleaned shape", X.shape, y.shape)

cleaned shape (178, 13) (178,)


In [15]:
X_train, X_val, X_test, y_train, y_val, y_test, encoder = split_data(X, y)
X_train, X_val, X_test = preprocess_features(
    X_train, X_val, X_test, numeric, categorical
)

X_train_tensor = torch.from_numpy(X_train).float()
X_val_tensor   = torch.from_numpy(X_val).float()
X_test_tensor  = torch.from_numpy(X_test).float()

X_train_val_tensor = torch.cat([X_train_tensor, X_val_tensor], dim=0)
X_train_val = X_train_val_tensor.detach().cpu().numpy()


print("split shape: X_train, X_val, X_test, X_train_val ",X_train.shape, X_val.shape, X_test.shape, X_train_val.shape)


split shape: X_train, X_val, X_test, X_train_val  (106, 13) (36, 13) (36, 13) (142, 13)
